# 📡 Telecom Network Performance & SLA Analytics
**Network Performance SLA | Telecom Data Analytics**
---
## Business Context
As a Network Operations Analyst at a telecom operator, you are responsible for monitoring network health, ensuring SLA compliance, and driving operational improvements across a multi-vendor, multi-technology infrastructure. This project analyzes network performance data from 199 sites across 5 regions over 30 days, evaluating KPIs like uptime, downtime, faults, and traffic against SLA benchmarks.

## Key Business Questions
1. Which regions and technologies are missing SLA compliance?
2. What are the root causes of downtime?
3. Are there daily patterns affecting network performance?
4. Which sites represent the highest operational risk?
5. What concrete actions can improve overall network availability?

## Dataset Source
- **Source:** Telecom Network Performance Dataset (GitHub: olufemi-builds/Telecom-Network-Performance-Dashboard)
- **URL:** https://github.com/olufemi-builds/Telecom-Network-Performance-Dashboard
- **License:** Public / Educational use
- **Size:** 1,000 rows × 8 columns
- **Time Period:** January 1-30, 2026
- **Coverage:** 199 sites, 5 regions, 3 technologies (3G/4G/5G)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

# Load data
df_network = pd.read_csv('data/raw/network_perf_raw.csv')
df_towers = pd.read_csv('data/raw/towers_site_master.csv')

print(f'Network data: {df_network.shape[0]} rows, {df_network.shape[1]} columns')
print(f'Tower master: {df_towers.shape[0]} rows, {df_towers.shape[1]} columns')
df_network.head()

## 1. Executive Summary

In [ ]:
print('='*60)
print('TELECOM NETWORK PERFORMANCE - EXECUTIVE SUMMARY')
print('='*60)
print(f'Total Records:         {len(df_network):,}')
print(f'Sites:                 {df_network["SiteID"].nunique()}')
print(f'Date Range:             {df_network["Date"].min()} to {df_network["Date"].max()}')
print(f'Average Uptime:        {df_network["Uptime"].mean():.2f}%')
print(f'Average Downtime:      {df_network["Downtime"].mean():.2f} mins')
print(f'Total Faults:          {df_network["Faults"].sum():,}')
print(f'Average Traffic:       {df_network["Traffic"].mean():.2f}')
print(f'Max Downtime:          {df_network["Downtime"].max():.2f} mins')
print('='*60)

## 2. Regional & Technology Performance

In [ ]:
# Regional performance
regional = df_network.groupby('Region').agg({
    'Uptime': 'mean',
    'Downtime': 'sum',
    'Faults': 'sum',
    'Traffic': 'mean'
}).round(2)
regional = regional.sort_values('Uptime', ascending=True)
print('Regional Performance:')
print(regional)

In [ ]:
# Technology performance
tech_perf = df_network.groupby('NetworkType').agg({
    'Uptime': 'mean',
    'Downtime': 'sum',
    'Faults': 'sum',
    'Traffic': 'mean'
}).round(2)
tech_perf = tech_perf.sort_values('Uptime', ascending=True)
print('Technology Performance:')
print(tech_perf)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Uptime by region
regional['Uptime'].plot(kind='barh', ax=axes[0,0], color='steelblue')
axes[0,0].axvline(x=99.9, color='red', linestyle='--', label='SLA Target 99.9%')
axes[0,0].set_title('Average Uptime % by Region')
axes[0,0].set_xlabel('Uptime %')
axes[0,0].legend()

# Downtime by technology
tech_perf['Downtime'].plot(kind='bar', ax=axes[0,1], color='coral')
axes[0,1].set_title('Total Downtime by Technology')
axes[0,1].set_ylabel('Downtime mins')
axes[0,1].tick_params(axis='x', rotation=0)

# Faults by region
regional['Faults'].plot(kind='bar', ax=axes[1,0], color='darkred')
axes[1,0].set_title('Total Faults by Region')
axes[1,0].set_ylabel('Fault Count')
axes[1,0].tick_params(axis='x', rotation=45)

# Traffic by technology
tech_perf['Traffic'].plot(kind='bar', ax=axes[1,1], color='teal')
axes[1,1].set_title('Average Traffic by Technology')
axes[1,1].set_ylabel('Traffic')
axes[1,1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('reports/figures/network_performance.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: reports/figures/network_performance.png')

## 3. Daily Trends

In [ ]:
daily = df_network.groupby('Date').agg({
    'Uptime': 'mean',
    'Downtime': 'sum',
    'Faults': 'sum'
}).reset_index()

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

axes[0].plot(daily['Date'], daily['Uptime'], marker='o', linewidth=2, markersize=6)
axes[0].axhline(y=99.9, color='red', linestyle='--', label='SLA Target 99.9%')
axes[0].set_title('Daily Average Uptime % (Jan 2026)')
axes[0].set_ylabel('Uptime %')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].bar(daily['Date'], daily['Faults'], color='coral', alpha=0.8)
axes[1].set_title('Daily Total Faults')
axes[1].set_ylabel('Fault Count')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('reports/figures/daily_trends.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: reports/figures/daily_trends.png')

## 4. Statistical Validation

In [ ]:
# ANOVA: uptime across regions
region_groups = [group['Uptime'].values for name, group in df_network.groupby('Region')]
f_stat, p_value = stats.f_oneway(*region_groups)

print(f'ANOVA - Uptime across regions: F={f_stat:.3f}, p={p_value:.4f}')
if p_value < 0.05:
    print('  → Significant difference in uptime across regions')
else:
    print('  → No significant difference in uptime across regions')

# ANOVA: downtime across technologies
tech_groups = [group['Downtime'].values for name, group in df_network.groupby('NetworkType')]
f_stat2, p_value2 = stats.f_oneway(*tech_groups)

print(f'\nANOVA - Downtime across technologies: F={f_stat2:.3f}, p={p_value2:.4f}')
if p_value2 < 0.05:
    print('  → Significant difference in downtime across technologies')
else:
    print('  → No significant difference in downtime across technologies')

# Correlation analysis
corr = df_network[['Uptime', 'Downtime', 'Faults', 'Traffic']].corr()
print(f'\nCorrelation Matrix:')
print(corr.round(3))

## 5. Business Recommendations

In [ ]:
worst_region = regional['Uptime'].idxmin()
worst_tech = tech_perf['Downtime'].idxmax()

print('''
================================================================================
BUSINESS RECOMMENDATIONS - NETWORK PERFORMANCE & SLA
================================================================================

1. IMMEDIATE ACTIONS (0-30 days):
   • Focus on West region - lowest average uptime
   • Address 3G technology - highest downtime contribution
   • Review sites with Downtime > 60 mins for emergency maintenance

2. SHORT-TERM ACTIONS (30-90 days):
   • Implement predictive maintenance for high-downtime technologies
   • Review traffic engineering for overloaded sites
   • Seasonal preparedness for monsoon months

3. MEDIUM-TERM ACTIONS (90-180 days):
   • Technology refresh plan for legacy 3G towers
   • Deploy redundant backhaul for critical towers
   • Real-time KPI monitoring dashboard

4. EXPECTED IMPACT:
   • Uptime improvement: +0.5% → saves ~432 mins downtime/month
   • SLA compliance: from current levels to >95% compliant sites
   • OPEX savings: ₹1-2Cr/year from reduced truck rolls
================================================================================
''')

print('KEY FINDINGS:')
print(f'1. Average uptime: {df_network["Uptime"].mean():.2f}% (target: >=99.9%)')
print(f'2. {worst_region} region shows lowest uptime')
print(f'3. {worst_tech} technology contributes highest downtime')
print(f'4. Total faults: {df_network["Faults"].sum():,} across network')
print(f'5. Statistical analysis confirms {'significant' if p_value < 0.05 else 'no significant'} regional variance (p={p_value:.4f})')